# Legal Research with IndianConstitution

This notebook demonstrates research-oriented workflows:

1. **Cross-reference network** analysis
2. **Fundamental Rights** deep-dive
3. **Article comparison** and structural analysis
4. **Data export** for external tools
5. **Statistical overview** of the Constitution

**Requirements:**
```bash
pip install "indianconstitution[data]"
```

In [ ]:
# Uncomment to install:
# !pip install -q "indianconstitution[data]"

In [ ]:
import networkx as nx
import pandas as pd

from indianconstitution import get_constitution

ic = get_constitution()
print(f"Loaded: {ic}")

## 1. Structural Overview

In [ ]:
df = ic.to_dataframe()
print(f"Total articles: {len(df)}")
print(f"\nArticles per Part:")
part_counts = df["part"].value_counts().sort_index()
for part, count in part_counts.items():
    if part is not None:
        print(f"  Part {int(part):>3d}: {count:>3d} articles")

In [ ]:
# Word count analysis
df["word_count"] = df["content"].apply(lambda x: len(str(x).split()))

print(f"Total words in Constitution: {df['word_count'].sum():,}")
print(f"Average words per article:   {df['word_count'].mean():.0f}")
print(f"Median words per article:    {df['word_count'].median():.0f}")

print(f"\nTop 10 longest articles:")
longest = df.nlargest(10, "word_count")[["number", "title", "word_count"]]
longest

## 2. Fundamental Rights Deep-Dive (Part III)

Articles 12–35 form the bedrock of citizen protections.

In [ ]:
fr_articles = [a for a in ic.articles if a.part == 3]
print(f"Fundamental Rights articles: {len(fr_articles)}")

for a in fr_articles:
    print(f"  Art. {a.number:>4s}: {a.title}")

In [ ]:
# Which Fundamental Rights articles are most referenced?
G = ic.get_graph()
fr_numbers = {str(a.number) for a in fr_articles}

fr_in_degree = {}
for node in fr_numbers:
    if node in G:
        fr_in_degree[node] = G.in_degree(node)

sorted_fr = sorted(fr_in_degree.items(), key=lambda x: x[1], reverse=True)
print("Most cited Fundamental Rights articles:")
for art_num, degree in sorted_fr[:10]:
    a = ic.get_article(art_num)
    print(f"  Art. {art_num}: {a.title if a else 'N/A'}  (cited by {degree} articles)")

## 3. Cross-Reference Network Analysis

In [ ]:
G = ic.get_graph()

print(f"Nodes (articles): {G.number_of_nodes()}")
print(f"Edges (references): {G.number_of_edges()}")
print(f"Density: {nx.density(G):.6f}")
print(f"Average clustering: {nx.average_clustering(G.to_undirected()):.4f}")

# Connected components (undirected)
components = list(nx.connected_components(G.to_undirected()))
print(f"Connected components: {len(components)}")
print(f"Largest component: {len(max(components, key=len))} articles")

In [ ]:
# PageRank — identifies constitutionally "important" articles
top_central = ic.get_central_articles(limit=15)

print("Top 15 articles by PageRank:")
for art_num, score in top_central:
    a = ic.get_article(art_num)
    print(f"  Art. {art_num:>4s}: {a.title if a else 'N/A':50s} (PR: {score:.6f})")

In [ ]:
# Deep-dive: Article 32 (Right to Constitutional Remedies)
related = ic.get_related_articles("32")

print("Article 32 — Right to Constitutional Remedies")
print(f"\n  References {len(related['references'])} articles: {related['references']}")
print(f"  Referenced by {len(related['referenced_by'])} articles: {related['referenced_by']}")

# Explore what Article 32 references
for ref in related["references"]:
    a = ic.get_article(ref)
    print(f"\n  \u2192 Art. {ref}: {a.title if a else 'N/A'}")

## 4. Research Export

Export the full corpus for analysis in external tools (R, SPSS, etc.).

In [ ]:
# Export to multiple formats
ic.export("json", "research_corpus.json")
ic.export("csv", "research_corpus.csv")
ic.export("markdown", "research_corpus.md")

print("\u2713 Exported to JSON, CSV, and Markdown")

# Also export the edge list for network tools (Gephi, etc.)
nx.write_edgelist(G, "constitution_edges.txt")
print("\u2713 Exported edge list for network tools")

In [ ]:
# Build a custom research dataset
research_df = df[["number", "title", "content", "part"]].copy()
research_df["word_count"] = research_df["content"].apply(lambda x: len(str(x).split()))
research_df["char_count"] = research_df["content"].apply(len)

# Add graph metrics
centrality = nx.degree_centrality(G)
pagerank = nx.pagerank(G)
research_df["degree_centrality"] = research_df["number"].map(centrality)
research_df["pagerank"] = research_df["number"].map(pagerank)

research_df.to_csv("constitution_research_dataset.csv", index=False)
print(f"\u2713 Research dataset: {research_df.shape}")
research_df.head(10)